# CUSTOMERS - INCREMENTAL LOAD WITH AUTO LOADER

## CREATED BY: RAHUL M
## CREATED DATE: 20260822
## DESCRIPTION: Incremental ingestion from landing using Auto Loader

### AUTO LOADER BENEFITS:
* **Incremental Processing**: Only new files are processed
* **Automatic Schema Evolution**: Handles schema changes gracefully
* **Checkpoint Management**: Tracks processed files automatically
* **No Duplicates**: Ensures each file is processed exactly once
* **Cost Efficient**: Reduces compute by avoiding full re-scans

In [0]:
%run /Workspace/Users/rms181800@gmail.com/AZURE_B3_PROJECT_AUG/FUNCTIONS/functions

In [0]:
from pyspark.sql.functions import current_timestamp

# Auto Loader Configuration
source_path = "/Volumes/retail_project_b3/landing/raw_data/"
target_path = "/Volumes/retail_project_b3/bronze/customer/"
checkpoint_path = "/Volumes/retail_project_b3/bronze/_checkpoints/customers"
schema_path = "/Volumes/retail_project_b3/bronze/_checkpoints/customers_schema"

# Read stream with Auto Loader
df_customer_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .option("cloudFiles.useNotifications", "false") \
    .load(f"{source_path}olist_customers_dataset.csv")

print("✅ Auto Loader stream initialized for customers")
print(f"📁 Source: {source_path}")
print(f"📁 Target: {target_path}")

In [0]:
# Add ingestion timestamp
df_customer_with_audit = df_customer_stream.withColumn("ingest_ts", current_timestamp())

print("✅ Audit timestamp added")

In [0]:
# Write stream with checkpoint (Incremental)
query = df_customer_with_audit.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path) \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start(target_path)

# Wait for completion
query.awaitTermination()

# Display results
if query.lastProgress:
    records_processed = query.lastProgress.get('numInputRows', 0)
    print(f"✅ Incremental load completed")
    print(f"📊 Records processed: {records_processed}")
    print(f"⏱️  Batch duration: {query.lastProgress.get('batchDuration', 0)} ms")
else:
    print("✅ No new files to process")